In [1]:
# Final Classifier model - rf and xgboost 
# For each cluster we train and tune a rf model that classifies rain into three categories: 1. No (<0.1 mm)
# 2. Light (<= 8.4 mm)
# 3. Heavy (> 8.4 mm)

# Workflow:
# 1. Load the complete df, format and create a prec intensity column (target)
# 2. Split df by clusters and run a for loop
# 3. Convert into np array for sklearn fit
# 4. Fit and tune a random forest using cv
# 5. Get validation metrics from training stations
# 6. Get validation metrics from validation stations 
# 7. Save trained model using joblib
# 8. Visualize final validation metrics for all clusters

In [2]:
import os
import numpy as np
import pandas as pd
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

train_data_path = r"..\Data\Processed\Classification_Regression\final_train_df.csv"
test_data_path  = r"..\Data\Processed\Classification_Regression\final_test_df.csv"
class_metrics_dir = r"..\Data\Processed\Classification_Regression\Classification_metrics"
class_models_dir  = r"..\Data\Processed\Classification_Regression\Classification_models"  

os.makedirs(class_metrics_dir, exist_ok=True)
os.makedirs(class_models_dir,  exist_ok=True)

random_state_num = 64
target_variable  = "intensity_prec"
n_cores          =  2

In [3]:
# %%
train_df = pd.read_csv(train_data_path)
test_df  = pd.read_csv(test_data_path)   

season_dict = {'winter': 1,
                'pre_monsoon': 2, 
                'monsoon': 3,
                'post_monsoon': 4            
                }

train_df['season_num'] = train_df['season'].map(season_dict)
test_df['season_num'] = test_df['season'].map(season_dict)

conditions_train = [
    (train_df['insitu_prec'] < 0.1),
    (train_df['insitu_prec'] >= 0.1) 
]
conditions_test = [
    (test_df['insitu_prec'] < 0.1),
    (test_df['insitu_prec'] >= 0.1) ]

intensity_class = [0, 1]

train_df[target_variable] = np.select(conditions_train, intensity_class, default=np.nan)
test_df[target_variable]  = np.select(conditions_test,  intensity_class, default=np.nan)  # was using undefined `conditions`

cluster_list = sorted(train_df['cluster'].unique().tolist())
print(f"Clusters found: {cluster_list}")

Clusters found: [0.0, 1.0, 2.0, 3.0]


In [ ]:
# %%
def get_param_grids():
    param_dist_rf = {
        'n_estimators':     [50, 100, 200, 500, 1000],
        'max_features':     ['sqrt', 'log2'],
        'max_depth':        [None, 10, 20, 30, 50],
        'min_samples_split':[2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    }
    param_dist_xgb = {
        'n_estimators':  [50, 100, 200, 500, 1000],
        'max_depth':     [3, 5, 7, 10],
        'learning_rate': [0.01, 0.05, 0.1, 0.2],
        'subsample':     [0.6, 0.8, 1.0],
        'colsample_bytree': [0.6, 0.8, 1.0],
        'gamma':         [0, 0.1, 0.5]
    }
    return param_dist_rf, param_dist_xgb


def fit_and_evaluate(model_type, X_train, y_train, X_test, y_test,
                     cluster, random_state_num, n_cores):
    """
    Fits RF or XGBoost classifier with RandomizedSearchCV.
    Returns best model, train accuracy, test accuracy, best params.
    """
    param_dist_rf, param_dist_xgb = get_param_grids()

    if model_type == 'rf':
        base_model = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('clf',     RandomForestClassifier(random_state=random_state_num, n_jobs=n_cores))
        ])
        param_dist = {f'clf__{k}': v for k, v in param_dist_rf.items()}
    elif model_type == 'xgb':
        base_model = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('clf',     XGBClassifier(random_state=random_state_num, n_jobs=n_cores,
                                    eval_metric='mlogloss', verbosity=0))
        ])
        param_dist = {f'clf__{k}': v for k, v in param_dist_xgb.items()}
    else:
        raise ValueError(f"model_type must be 'rf' or 'xgb', got '{model_type}'")

    search = RandomizedSearchCV(
        estimator=base_model,
        param_distributions=param_dist,
        n_iter=50,
        cv=5,
        verbose=1,
        random_state=random_state_num,
        n_jobs=n_cores,
        scoring='accuracy'
    )
    search.fit(X_train, y_train)
    best_model = search.best_estimator_

    train_preds    = best_model.predict(X_train)
    test_preds     = best_model.predict(X_test)
    train_accuracy = accuracy_score(y_train, train_preds)
    test_accuracy  = accuracy_score(y_test,  test_preds)

    print(f"\n[Cluster {cluster} | {model_type.upper()}]")
    print(f"  Best params      : {search.best_params_}")
    print(f"  CV accuracy      : {search.best_score_:.4f}")
    print(f"  Train accuracy   : {train_accuracy:.4f}")
    print(f"  Test accuracy    : {test_accuracy:.4f}")
    all_class_names = {0: 'No Rain', 1: 'Rain'}
    present_labels = sorted(set(y_test) | set(test_preds))
    t_names = [all_class_names[c] for c in present_labels]
    print(classification_report(y_test, test_preds, labels=present_labels, target_names=t_names))

    return best_model, train_accuracy, test_accuracy, search.best_params_

In [5]:
# %%
results = []  

for cluster in cluster_list:
    print(f"\n{'='*55}\nProcessing Cluster {cluster}\n{'='*55}")

    # 2. Split by cluster
    cluster_train_df = train_df[train_df['cluster'] == cluster].copy()
    cluster_test_df  = test_df[test_df['cluster']  == cluster].copy()
    cluster_train_df = cluster_train_df.dropna(subset=[target_variable])
    cluster_test_df  = cluster_test_df.dropna(subset=[target_variable])

    # 3. Build feature matrices — drop non-feature columns
    cols_to_drop = [target_variable, 'cluster', 'date', 'insitu_prec', 'season']
    X_train = cluster_train_df.drop(columns=cols_to_drop).values
    X_test  = cluster_test_df.drop(columns=cols_to_drop).values
    y_train = cluster_train_df[target_variable].values.astype(int)
    y_test  = cluster_test_df[target_variable].values.astype(int)

    feature_list = cluster_train_df.drop(columns=cols_to_drop).columns.tolist()

    for model_type in ['rf', 'xgb']:
        # 4 + 5 + 6. Fit, tune, evaluate
        best_model, train_acc, test_acc, best_params = fit_and_evaluate(
            model_type, X_train, y_train, X_test, y_test,
            cluster, random_state_num, n_cores
        )

        # 7. Save model
        model_filename = f"cluster{cluster}_{model_type}_classifier_2classes.pkl"
        model_path     = os.path.join(class_models_dir, model_filename)
        joblib.dump(best_model, model_path)
        print(f"  Model saved → {model_path}")

        # Collect metrics
        results.append({
            'cluster':        cluster,
            'model':          model_type.upper(),
            'train_accuracy': round(train_acc, 4),
            'test_accuracy':  round(test_acc,  4),
        })


Processing Cluster 0.0
Fitting 5 folds for each of 50 candidates, totalling 250 fits

[Cluster 0.0 | RF]
  Best params      : {'clf__n_estimators': 100, 'clf__min_samples_split': 5, 'clf__min_samples_leaf': 1, 'clf__max_features': 'log2', 'clf__max_depth': 10}
  CV accuracy      : 0.8243
  Train accuracy   : 0.8685
  Test accuracy    : 0.8208
              precision    recall  f1-score   support

     No Rain       0.87      0.89      0.88      7960
        Rain       0.65      0.62      0.63      2658

    accuracy                           0.82     10618
   macro avg       0.76      0.75      0.76     10618
weighted avg       0.82      0.82      0.82     10618

  Model saved → ..\Data\Processed\Classification_Regression\Classification_models\cluster0.0_rf_classifier_2classes.pkl
Fitting 5 folds for each of 50 candidates, totalling 250 fits

[Cluster 0.0 | XGB]
  Best params      : {'clf__subsample': 0.6, 'clf__n_estimators': 100, 'clf__max_depth': 5, 'clf__learning_rate': 0.05, 'clf

In [6]:
# %%
# 8. Validation metrics table for all clusters
results_df = pd.DataFrame(results)

# ── Compute recall generalisation gap from stored results ─────────────────────
# results list already has train/test accuracy; re-predict on both splits
# to extract Rain-class recall specifically.
from sklearn.metrics import recall_score

recall_rows = []
for cluster in cluster_list:
    cluster_train_df = train_df[train_df['cluster'] == cluster].dropna(subset=[target_variable])
    cluster_test_df  = test_df[test_df['cluster']   == cluster].dropna(subset=[target_variable])

    cols_to_drop = [target_variable, 'cluster', 'date', 'insitu_prec', 'season']
    X_train = cluster_train_df.drop(columns=cols_to_drop).values
    X_test  = cluster_test_df.drop(columns=cols_to_drop).values
    y_train = cluster_train_df[target_variable].values.astype(int)
    y_test  = cluster_test_df[target_variable].values.astype(int)

    for model_type in ['rf', 'xgb']:
        model_path = os.path.join(class_models_dir,
                                  f"cluster{cluster}_{model_type}_classifier_2classes.pkl")
        if not os.path.exists(model_path):
            continue
        model = joblib.load(model_path)

        train_recall = recall_score(y_train, model.predict(X_train), pos_label=1)
        test_recall  = recall_score(y_test,  model.predict(X_test),  pos_label=1)

        recall_rows.append({
            'cluster':       cluster,
            'model':         model_type.upper(),
            'train_recall':  round(train_recall, 4),
            'test_recall':   round(test_recall,  4),
            'recall_gap':    round(train_recall - test_recall, 4),
        })

recall_df = pd.DataFrame(recall_rows)

# ── Merge accuracy and recall gaps ───────────────────────────────────────────
summary_df = results_df.copy()
summary_df['accuracy_gap'] = (summary_df['train_accuracy']
                               - summary_df['test_accuracy']).round(4)
summary_df = summary_df.merge(recall_df, on=['cluster', 'model'], how='left')
summary_df = summary_df[['cluster', 'model',
                          'train_accuracy', 'test_accuracy', 'accuracy_gap',
                          'train_recall',   'test_recall',   'recall_gap']]

# ── Average rows ──────────────────────────────────────────────────────────────
avg_cols = ['train_accuracy', 'test_accuracy', 'accuracy_gap',
            'train_recall',   'test_recall',   'recall_gap']
avg_rows = (
    summary_df.groupby('model')[avg_cols]
    .mean().round(4).reset_index()
    .assign(cluster='AVERAGE')
)
summary_df = pd.concat([summary_df, avg_rows], ignore_index=True)
summary_df = summary_df[['cluster', 'model',
                          'train_accuracy', 'test_accuracy', 'accuracy_gap',
                          'train_recall',   'test_recall',   'recall_gap']]

print("\n===== Validation Metrics Summary =====")
print(summary_df.to_string(index=False))

summary_path = os.path.join(class_metrics_dir, "validation_metrics_summary_2classes.csv")
summary_df.to_csv(summary_path, index=False)
print(f"\nSummary saved → {summary_path}")


===== Validation Metrics Summary =====
cluster model  train_accuracy  test_accuracy  accuracy_gap  train_recall  test_recall  recall_gap
    0.0    RF          0.8685         0.8208        0.0477        0.7011       0.6196      0.0815
    0.0   XGB          0.8429         0.8208        0.0221        0.6563       0.6181      0.0382
    1.0    RF          0.9368         0.8158        0.1210        0.8964       0.6723      0.2241
    1.0   XGB          0.8634         0.8082        0.0552        0.7932       0.6523      0.1408
    2.0    RF          0.9012         0.8396        0.0616        0.5818       0.3438      0.2380
    2.0   XGB          0.8864         0.8413        0.0451        0.5553       0.3797      0.1756
    3.0    RF          0.8935         0.7996        0.0939        0.7034       0.5463      0.1571
    3.0   XGB          0.8268         0.8042        0.0226        0.5576       0.5549      0.0027
AVERAGE    RF          0.9000         0.8190        0.0810        0.7207      